# Annotation naive

This notebook is the naive model-screening experiment. It sends the same
full-codebook prompt (`P0`) and the same ten validation dialogues to every
model. No model-specific prompt tuning is used, so the comparison tests how
well each model follows the annotation task natively.

Results are cached under:

```text
extension/artifacts/extraction_cache/naive_testing/{model}/P0/{dialogue_id}.json
```


## 1. Load the validation dataset

The human validation set supplies the dialogue text sent to the models and
the gold annotations used for scoring. The prompt receives only the
conversation and unit names; gold labels and adjudication fields remain
hidden.


In [1]:
import os
import sys
from pathlib import Path

here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / 'extension' / 'artifacts').exists():
        os.chdir(candidate)
        break
else:
    raise FileNotFoundError('Run this notebook from inside the Experiment_1 repository.')

sys.path.insert(0, str(Path.cwd()))

if not os.environ.get('OPENROUTER_API_KEY') and Path('.env').exists():
    for line in Path('.env').read_text().splitlines():
        if line.strip().startswith('OPENROUTER_API_KEY='):
            os.environ['OPENROUTER_API_KEY'] = line.split('=', 1)[1].strip().strip('\"').strip("'")
            break

print('repository:', Path.cwd())
print('OPENROUTER_API_KEY set:', bool(os.environ.get('OPENROUTER_API_KEY')))


repository: /Users/tandon.utsav2/Desktop/Experiment_1
OPENROUTER_API_KEY set: True


In [2]:
from extension.scripts.data_management.load_annotation_data import load_dataset
from extension.scripts.annotation import extraction, prompt_loader, scoring

VALIDATION_PATH = Path('extension/artifacts/annotation_dev_val_and_eval_sets/validation_set.csv')
CACHE_NAMESPACE = 'naive_testing'
PROMPT = 'P0'
N_DIALOGUES = 10

gold = load_dataset(VALIDATION_PATH)
dialogues = extraction.dialogues_from(gold, split=CACHE_NAMESPACE)
test_dialogues = dialogues[:N_DIALOGUES]
dialogue_ids = [dialogue['dialogue_id'] for dialogue in test_dialogues]

assert PROMPT in prompt_loader.list_prompts(), prompt_loader.list_prompts()
assert len(test_dialogues) == N_DIALOGUES
print(f'loaded {len(gold)} annotated units across {len(dialogues)} dialogues')
print('test dialogue ids:', dialogue_ids)
print('cache root:', Path('extension/artifacts/extraction_cache') / CACHE_NAMESPACE)


loaded 544 annotated units across 78 dialogues
test dialogue ids: [1, 21, 35, 79, 143, 178, 255, 270, 275, 289]
cache root: extension/artifacts/extraction_cache/naive_testing


## 2. Model configuration

All models are held in one configuration list. `reasoning_effort` is used
when OpenRouter exposes an effort selector; `reasoning_enabled=True` is used
when the provider supports reasoning but manages its own reasoning budget.
Models without either control receive no reasoning field. Temperature is
zero for every request.


In [3]:
import pandas as pd

MODELS = [
    {'group': 'large', 'name': 'GLM 5.3', 'model': 'z-ai/glm-5.3', 'reasoning_effort': 'max', 'reasoning_enabled': None, 'reasoning_setting': 'max'},
    {'group': 'large', 'name': 'Kimi K3', 'model': 'moonshotai/kimi-k3', 'reasoning_effort': 'max', 'reasoning_enabled': None, 'reasoning_setting': 'max'},
    {'group': 'medium', 'name': 'Qwen3.5 122B-A10B', 'model': 'qwen/qwen3.5-122b-a10b', 'reasoning_effort': None, 'reasoning_enabled': True, 'reasoning_setting': 'enabled (model-managed)'},
    {'group': 'medium', 'name': 'GPT-OSS 120B', 'model': 'openai/gpt-oss-120b', 'reasoning_effort': 'high', 'reasoning_enabled': None, 'reasoning_setting': 'high'},
    {'group': 'medium', 'name': 'Nemotron 3 Super 120B-A12B', 'model': 'nvidia/nemotron-3-super-120b-a12b', 'reasoning_effort': 'medium', 'reasoning_enabled': None, 'reasoning_setting': 'medium'},
    {'group': 'small', 'name': 'Qwen3.5 35B-A3B', 'model': 'qwen/qwen3.5-35b-a3b', 'reasoning_effort': None, 'reasoning_enabled': True, 'reasoning_setting': 'enabled (model-managed)'},
    {'group': 'small', 'name': 'Qwen3.5 27B', 'model': 'qwen/qwen3.5-27b', 'reasoning_effort': None, 'reasoning_enabled': True, 'reasoning_setting': 'enabled (model-managed)'},
    {'group': 'small', 'name': 'GPT-OSS 20B', 'model': 'openai/gpt-oss-20b', 'reasoning_effort': 'high', 'reasoning_enabled': None, 'reasoning_setting': 'high'},
    {'group': 'small', 'name': 'Gemma 3 12B', 'model': 'google/gemma-3-12b-it', 'reasoning_effort': None, 'reasoning_enabled': None, 'reasoning_setting': 'unsupported'},
    {'group': 'small', 'name': 'Qwen3.5 9B', 'model': 'qwen/qwen3.5-9b', 'reasoning_effort': None, 'reasoning_enabled': True, 'reasoning_setting': 'enabled (model-managed)'},
    {'group': 'small', 'name': 'Gemma 3 4B', 'model': 'google/gemma-3-4b-it', 'reasoning_effort': None, 'reasoning_enabled': None, 'reasoning_setting': 'unsupported'},
]

MAX_WORKERS = 2
extraction.TEMPERATURE = 0.0
pd.DataFrame(MODELS)[['group', 'name', 'model', 'reasoning_setting']]


,group,name,model,reasoning_setting
0,large,GLM 5.3,z-ai/glm-5.3,max
1,large,Kimi K3,moonshotai/kimi-k3,max
2,medium,Qwen3.5 122B-A10B,qwen/qwen3.5-122b-a10b,enabled (model-managed)
3,medium,GPT-OSS 120B,openai/gpt-oss-120b,high
4,medium,Nemotron 3 Super 120B-A12B,nvidia/nemotron-3-super-120b-a12b,medium
5,small,Qwen3.5 35B-A3B,qwen/qwen3.5-35b-a3b,enabled (model-managed)
6,small,Qwen3.5 27B,qwen/qwen3.5-27b,enabled (model-managed)
7,small,GPT-OSS 20B,openai/gpt-oss-20b,high
8,small,Gemma 3 12B,google/gemma-3-12b-it,unsupported
9,small,Qwen3.5 9B,qwen/qwen3.5-9b,enabled (model-managed)


## 3. Extraction

Leave `SKIP = True` when reviewing cached results. Set it to `False` only
when inference is intended. The shared extraction script reuses valid cache
records and requests only missing or invalid records. Consequently, changing
`SKIP` can incur API costs and will retry model-limitation failures as well as
transport failures.


In [4]:
SKIP = True

if SKIP:
    print('Extraction skipped; existing cache files are unchanged.')
else:
    if not os.environ.get('OPENROUTER_API_KEY'):
        raise RuntimeError('Set OPENROUTER_API_KEY before running extraction.')

    for config in MODELS:
        print(
            f"{config['group'].upper()} | {config['name']} | "
            f"reasoning={config['reasoning_setting']}"
        )
        extraction.generate_annotations(
            PROMPT,
            config['model'],
            test_dialogues,
            max_workers=MAX_WORKERS,
            reasoning_effort=config['reasoning_effort'],
            reasoning_enabled=config['reasoning_enabled'],
        )


Extraction skipped; existing cache files are unchanged.


## 4. Cached results

The five reported metrics are calculated by the shared scoring script. Overall
`kappa` is nominal Cohen's kappa across every valid P/A/N family-turn cell.
`valid_rate` uses all ten requested dialogues. The remaining metrics use only
valid annotations, so results based on a low valid rate are vulnerable to
selection bias and should not be compared directly with complete results.


In [5]:
score_rows = []
for config in MODELS:
    metrics = scoring.score_config(
        gold,
        config['model'],
        PROMPT,
        dialogue_ids,
        n_boot=0,
        split=CACHE_NAMESPACE,
    )
    score_rows.append({
        'group': config['group'],
        'model': config['name'],
        'valid_rate': metrics['valid_rate'],
        'macro_f1': metrics['macro_f1_P'],
        'accuracy': metrics['accuracy'],
        'alpha': metrics['alpha'],
        'kappa': metrics['kappa'],
    })

results = pd.DataFrame(score_rows)
display(results.round(3))


,group,model,valid_rate,macro_f1,accuracy,alpha,kappa
0,large,GLM 5.3,0.2,1.000,0.771,0.571,0.591
1,large,Kimi K3,1.0,0.502,0.852,0.618,0.619
2,medium,Qwen3.5 122B-A10B,0.5,0.252,0.737,0.213,0.218
3,medium,GPT-OSS 120B,1.0,0.311,0.573,0.180,0.247
4,medium,Nemotron 3 Super 120B-A12B,0.6,0.156,0.635,0.189,0.211
5,small,Qwen3.5 35B-A3B,0.2,0.467,0.814,0.434,0.440
6,small,Qwen3.5 27B,0.7,0.241,0.786,0.342,0.342
7,small,GPT-OSS 20B,0.8,0.114,0.615,0.109,0.127
8,small,Gemma 3 12B,0.8,0.183,0.338,-0.177,0.005
9,small,Qwen3.5 9B,0.0,NaN,NaN,NaN,NaN


## 5. Failure analysis and model choice

The cached run shows two distinct failure layers. A **format failure** means
the response could not pass the annotation schema and therefore contributes
only to `valid_rate`. A **semantic failure** means the response was valid JSON
but its labels disagreed with the human annotation.

| Model | Valid | Principal failure mode | Interpretation |
|---|---:|---|---|
| GLM 5.3 | 2/10 | Eight responses reached exactly 65,536 completion tokens, using 65,507--65,536 of them for reasoning, and ended without annotation JSON. In the two valid dialogues, comprehension presence was perfect, but the model produced eight false `A` labels across the other families. | The apparent macro F1 of `1.000` is selection-biased: only two dialogues survived, and comprehension was the only family with positive support. Accuracy (`0.771`) and alpha (`0.571`) are more informative but still describe only those two dialogues. |
| **Kimi K3** | **10/10** | No format failures. Its main semantic weakness was under-detecting wrong-operation errors and occasionally predicting principles where gold had none. | It followed the long codebook and output contract consistently while preserving strong label agreement. |
| Qwen3.5 122B-A10B | 5/10 | Two reasoning-only responses, two malformed JSON responses, and one resolution-invariant failure; valid outputs rarely used `A` and missed relevance. | Both completion reliability and label calibration were unstable. |
| GPT-OSS 120B | 10/10 | No format failures, but it heavily overpredicted `A`, under-detected wrong-operation errors, and invented principles/steps evidence. | High compliance, but weak calibration to the codebook's engagement and family boundaries. |
| Nemotron 3 Super 120B-A12B | 6/10 | Four reasoning-only responses; valid outputs missed relevance and wrong-operation errors and overcalled comprehension. | Long reasoning frequently displaced the required final answer, and the remaining labels were poorly separated. |
| Qwen3.5 35B-A3B | 2/10 | Seven reasoning-only responses and one malformed JSON response. | Its apparently high macro F1 is based on only two surviving dialogues and is therefore strongly selection-biased. |
| Qwen3.5 27B | 7/10 | Three reasoning-only responses; valid outputs missed relevance and underused `A`. | More reliable than the smaller Qwen variants, but still incomplete and semantically narrow. |
| GPT-OSS 20B | 8/10 | Two responses exhausted the output allowance during reasoning; valid outputs confused `A`/`N`, missed relevance and wrong-operation errors, and invented principles/steps evidence. | Reasonable format compliance did not yield dependable annotations. |
| Gemma 3 12B | 8/10 | Two responses violated thread-resolution invariants; valid outputs dramatically overpredicted `A` and missed most misconception presence. | It produced structured output but did not apply the codebook's evidence standard. |
| Qwen3.5 9B | 0/10 | Every response used its completion budget for reasoning and produced no final JSON. | The task and full codebook exceed this configuration's practical completion capacity. |
| Gemma 3 4B | 2/10 | Eight schema failures, including combined family names, long-form labels instead of `P/A/N`, missing units, and invalid resolution links. | The model could not reliably follow the output contract; metrics from two valid dialogues are not representative. |

### Conclusion

GLM 5.3 is not competitive in this configuration despite its displayed macro
F1 of `1.000`. Its 20% valid rate means that score is calculated from only
dialogues 35 and 143. The eight failures were model-completion failures, not
rate limits or connection errors: each request consumed the full 65,536-token
completion allowance almost entirely as reasoning and emitted no final JSON.
The run averaged 624 seconds and $0.284 per requested dialogue.

Among the included models with a 100% valid rate, Kimi K3 has the strongest
point estimates: macro F1 `0.502`, accuracy `0.852`, alpha `0.618`, and kappa
`0.619`, versus GPT-OSS 120B's `0.311`, `0.573`, `0.180`, and `0.247`. This
ten-dialogue screen therefore
supports Kimi as the strongest configuration retained in this comparison. The
sample is still too small to be treated as a final production evaluation, so
the larger validation run and qualitative reasoning audits remain important.
